# Step 4.03: Event study with a t=-2 reference year

This notebook is a presentation oriented variant of the event study
check. Instead of using the average of `t=-2` and `t=-1` as the
baseline, it sets `t=-2` equal to zero for each researcher and conference
event unit.

This makes the plot easier to read: `t=-1` shows the before PC change,
`t=0` shows the PC year change, and `t=+1,+2` show whether the change
persists or fades.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from datetime import date
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import IFrame, Markdown, display
from matplotlib import font_manager
from matplotlib.backends.backend_pdf import PdfPages

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_4_PREPARED = PROJECT / "step_4_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
MAIN_TEXT_FIGURES = PROJECT / "step_4_artifacts" / "main_text_figures"
T_MINUS_EVENT_FIGURES = (
    PROJECT / "step_4_artifacts" / "figures_t_minus_event_study"
)
REPORT_FIGURES = PROJECT / "report_latex" / "figures"
ensure_dirs(STEP_4_SUMMARY, MAIN_TEXT_FIGURES, T_MINUS_EVENT_FIGURES, REPORT_FIGURES)

EVENT_ROWS_IN = STEP_4_PREPARED / "career_age_event_window_rows.parquet"

T_MINUS_2_ROWS_OUT = (
    STEP_4_PREPARED / "t_minus_2_reference_event_window_rows.parquet"
)
T_MINUS_2_LOG_SUMMARY_OUT = (
    STEP_4_SUMMARY
    / "tminus2_icfp_log.csv"
)
T_MINUS_2_RAW_SUMMARY_OUT = (
    STEP_4_SUMMARY
    / "tminus2_icfp_raw.csv"
)
T_MINUS_2_FIGURE_RECORDS_OUT = (
    STEP_4_SUMMARY
    / "tminus2_figure_records.csv"
)

T_MINUS_2_LOG_FIGURE_OUT = (
    MAIN_TEXT_FIGURES / "t_minus_icfp_log.pdf"
)
T_MINUS_2_RAW_FIGURE_OUT = (
    MAIN_TEXT_FIGURES / "t_minus_icfp_raw.pdf"
)

REPORT_MAIN_EVENT_LOG_FIGURE_OUT = (
    REPORT_FIGURES / "event_study_icfp_log.pdf"
)
REPORT_MAIN_EVENT_RAW_FIGURE_OUT = (
    REPORT_FIGURES / "event_study_icfp_raw.pdf"
)
AGGREGATE_COMPARISON_FIGURE_OUT = (
    T_MINUS_EVENT_FIGURES / "aggregate_sample_comparison.pdf"
)
REPORT_AGGREGATE_COMPARISON_FIGURE_OUT = (
    REPORT_FIGURES / "aggregate_sample_comparison.pdf"
)

EVENT_TIMES = np.array([-2, -1, 0, 1, 2])
BOOTSTRAP_REPS = 2000

print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Run date: {date.today().isoformat()}")

Project folder: .
Run mode: fast
Run date: 2026-06-19


## 2. Plot style

In [2]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "font.size": 11,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "font.family": font_family,
        "text.usetex": True,
        "axes.linewidth": 0.8,
        "axes.edgecolor": "black",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

MAIN_COLOR = "#81c8be"
CONFERENCE_COLORS = {
    "Aggregate": "#6f6f6f",
    "POPL": "#81c8be",
    "ICFP": "#f5bde6",
    "OOPSLA": "#8aadf4",
    "PLDI before 2021": "#e5c890",
    "PLDI 2021 onward": "#e5c890",
}

## 3. Load the prepared balanced event window rows

In [3]:
rows = pd.read_parquet(EVENT_ROWS_IN)
rows = rows.loc[rows["is_isolated_first_pc_event_window"].eq(True)].copy()
rows["event_time"] = rows["event_time"].astype(int)
rows["citation_count"] = pd.to_numeric(rows["citation_count"], errors="coerce")
rows["log10_citations"] = np.log10(rows["citation_count"] + 1)
rows["plot_group"] = rows["conference"]
rows.loc[
    rows["conference"].eq("PLDI") & rows["year"].lt(2021),
    "plot_group",
] = "PLDI before 2021"
rows.loc[
    rows["conference"].eq("PLDI") & rows["year"].ge(2021),
    "plot_group",
] = "PLDI 2021 onward"
aggregate_groups = ["ICFP", "POPL", "PLDI 2021 onward"]

print(f"balanced event rows: {rows.shape[0]:,}")
print(f"event units: {rows['event_unit_id'].nunique():,}")
display(rows[["event_unit_id", "name", "conference", "year", "event_time", "citation_count"]].head())

balanced event rows: 2,015
event units: 403


,event_unit_id,name,conference,year,event_time,citation_count
0,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2021,-2,0.0
1,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2022,-1,0.0
2,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2023,0,0.0
3,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2024,1,1.0
4,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2025,2,0.0


## 4. Why the aggregate count is N=349

In [4]:
aggregate_counts = (
    rows.loc[rows["plot_group"].isin(aggregate_groups)]
    .drop_duplicates("event_unit_id")
    .groupby("plot_group")["event_unit_id"]
    .nunique()
    .sort_index()
    .rename("event_units")
    .reset_index()
)
aggregate_total = int(aggregate_counts["event_units"].sum())

print(
    "N=349 is the number of retained researcher--conference event units "
    "in the aggregate figure after restricting to ICFP, POPL, and "
    "PLDI 2021 onward."
)
print(
    "It is not a count of panel rows. Each retained event unit contributes "
    "five event times, so 349 event units become 1,745 event rows."
)
display(aggregate_counts)
print(f"Check: aggregate total = {aggregate_total}")

N=349 is the number of retained researcher--conference event units in the aggregate figure after restricting to ICFP, POPL, and PLDI 2021 onward.
It is not a count of panel rows. Each retained event unit contributes five event times, so 349 event units become 1,745 event rows.


,plot_group,event_units
0,ICFP,106
1,PLDI 2021 onward,138
2,POPL,138


Check: aggregate total = 382


## 5. Re reference the event window to t=-2

In [5]:
reference_values = (
    rows.loc[rows["event_time"].eq(-2)]
    .set_index("event_unit_id")[["citation_count", "log10_citations"]]
    .rename(
        columns={
            "citation_count": "reference_raw_citations_t_minus_2",
            "log10_citations": "reference_log10_citations_t_minus_2",
        }
    )
)

event_rows = rows.merge(
    reference_values,
    on="event_unit_id",
    how="left",
    validate="many_to_one",
)

event_rows["delta_from_t_minus_2_raw_citations"] = (
    event_rows["citation_count"]
    - event_rows["reference_raw_citations_t_minus_2"]
)
event_rows["delta_from_t_minus_2_log10_citations"] = (
    event_rows["log10_citations"]
    - event_rows["reference_log10_citations_t_minus_2"]
)
event_rows["event_study_baseline"] = "t_minus_2_reference"

check = event_rows.loc[event_rows["event_time"].eq(-2)]
assert np.allclose(check["delta_from_t_minus_2_raw_citations"].fillna(0), 0)
assert np.allclose(check["delta_from_t_minus_2_log10_citations"].fillna(0), 0)

display(
    event_rows[
        [
            "event_unit_id",
            "name",
            "conference",
            "event_time",
            "citation_count",
            "delta_from_t_minus_2_raw_citations",
            "delta_from_t_minus_2_log10_citations",
        ]
    ].head(10)
)

,event_unit_id,name,conference,event_time,citation_count,delta_from_t_minus_2_raw_citations,delta_from_t_minus_2_log10_citations
0,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,-2,0.0,0.0,0.000000
1,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,-1,0.0,0.0,0.000000
2,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,0,0.0,0.0,0.000000
3,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,1,1.0,1.0,0.301030
4,abhinavverma1|PLDI|2023,Abhinav Verma,PLDI,2,0.0,0.0,0.000000
5,adamchlipala|POPL|2021,Adam Chlipala,POPL,-2,16.0,0.0,0.000000
6,adamchlipala|POPL|2021,Adam Chlipala,POPL,-1,6.0,-10.0,-0.385351
7,adamchlipala|POPL|2021,Adam Chlipala,POPL,0,6.0,-10.0,-0.385351
8,adamchlipala|POPL|2021,Adam Chlipala,POPL,1,16.0,0.0,0.000000
9,adamchlipala|POPL|2021,Adam Chlipala,POPL,2,8.0,-8.0,-0.276206


## 6. Bootstrap summaries

In [6]:
def bootstrap_summary(frame, value_col, event_times=EVENT_TIMES, seed=1234):
    if frame.empty:
        return pd.DataFrame()

    matrix = (
        frame.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values=value_col,
            aggfunc="mean",
        )
        .reindex(columns=event_times)
        .to_numpy()
    )
    n_units = matrix.shape[0]
    if n_units == 0:
        return pd.DataFrame()

    means = np.nanmean(matrix, axis=0)
    rng = np.random.default_rng(seed)
    boot = np.empty((BOOTSTRAP_REPS, len(event_times)))
    for draw in range(BOOTSTRAP_REPS):
        idx = rng.choice(n_units, size=n_units, replace=True)
        boot[draw] = np.nanmean(matrix[idx], axis=0)

    summary = pd.DataFrame(
        {
            "event_time": event_times,
            "mean": means,
            "ci_lower": np.nanpercentile(boot, 2.5, axis=0),
            "ci_upper": np.nanpercentile(boot, 97.5, axis=0),
            "n_event_units": n_units,
        }
    )
    return summary


def broad_first_service_mask(frame):
    return frame["is_true_first_broad_service_in_observed_year_conf"].eq(True)


def assign_plot_group(frame):
    group = frame["conference"].copy()
    is_pldi = group.eq("PLDI")
    group.loc[is_pldi & frame["first_pc_year_conference"].lt(2021)] = (
        "PLDI before 2021"
    )
    group.loc[is_pldi & frame["first_pc_year_conference"].ge(2021)] = (
        "PLDI 2021 onward"
    )
    return group


event_rows["plot_group"] = assign_plot_group(event_rows)

## 6. Plot helpers

In [7]:
def draw_event_axis(
    ax,
    summary,
    color,
    title,
    ylabel,
    show_ylabel=True,
    show_xlabel=True,
):
    if summary.empty:
        ax.text(0.5, 0.5, "No event units", ha="center", va="center")
        ax.axis("off")
        return

    ax.axhline(0, color="0.78", lw=0.45, zorder=0)
    ax.axvline(0, color="black", lw=0.85, ls="--")
    ax.scatter(
        summary["event_time"],
        summary["mean"],
        color=color,
        s=24,
        zorder=3,
    )

    ci_rows = summary.loc[summary["event_time"].ne(-2)].copy()
    if not ci_rows.empty:
        yerr = np.vstack(
            [
                ci_rows["mean"].to_numpy() - ci_rows["ci_lower"].to_numpy(),
                ci_rows["ci_upper"].to_numpy() - ci_rows["mean"].to_numpy(),
            ]
        )
        ax.errorbar(
            ci_rows["event_time"],
            ci_rows["mean"],
            yerr=yerr,
            fmt="none",
            ecolor="black",
            elinewidth=0.85,
            capsize=3.2,
            capthick=0.85,
            zorder=4,
        )

    ax.set_xticks(EVENT_TIMES)
    if show_xlabel:
        ax.set_xlabel("Event time")
    else:
        ax.set_xlabel("")
    if show_ylabel:
        ax.set_ylabel(ylabel)
    else:
        ax.set_ylabel("")
    ax.set_title(title, loc="left")
    ax.grid(color="#DDDDDD", ls=":", lw=0.65)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_title(ax, title):
    ax.text(
        0.0,
        1.16,
        title,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.2,
        clip_on=False,
    )


def draw_hist_axis(
    ax,
    values,
    bins,
    color,
    title,
    xlabel,
    xlim=None,
    subtitle=None,
):
    values = pd.Series(values).dropna()
    if values.empty:
        ax.text(
            0.5,
            0.5,
            "No data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.axis("off")
        return

    counts, _, _ = ax.hist(
        values,
        bins=bins,
        color=color,
        alpha=0.22,
        edgecolor="white",
        linewidth=0.4,
    )
    max_count = max(float(np.max(counts)), 1.0)
    rng = np.random.default_rng(1234)
    point_y = rng.uniform(0.015 * max_count, 0.09 * max_count, len(values))
    ax.scatter(
        values,
        point_y,
        s=8,
        color="#333333",
        alpha=0.35,
        linewidth=0,
        zorder=3,
    )
    ax.axvline(values.median(), color="black", lw=1.1)
    ax.axvline(values.mean(), color=color, lw=1.2, ls="--")
    stats = f"median={values.median():.1f}, mean={values.mean():.1f}"
    if subtitle:
        add_panel_title(ax, f"{title}\n{subtitle}; {stats}")
    else:
        add_panel_title(ax, f"{title}\n{stats}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Event units")
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_ylim(0, max_count * 1.18)
    ax.grid(color="#DDDDDD", ls=":", lw=0.65)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_pre_pc_raw_axis(ax, rows, color):
    pre_rows = rows.loc[
        rows["event_time"].isin([-2, -1]),
        ["event_unit_id", "event_time", "citation_count"],
    ].copy()
    pre_rows["citation_count"] = pd.to_numeric(
        pre_rows["citation_count"],
        errors="coerce",
    )
    if pre_rows.empty:
        ax.text(
            0.5,
            0.5,
            "No data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.axis("off")
        return

    pivot = (
        pre_rows.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values="citation_count",
            aggfunc="mean",
        )
        .reindex(columns=[-2, -1])
    )

    positions = np.array([0, 1])
    data = [pivot[-2].dropna(), pivot[-1].dropna()]
    box = ax.boxplot(
        data,
        positions=positions,
        widths=0.34,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.1},
        boxprops={"facecolor": color, "alpha": 0.22, "edgecolor": "black"},
        whiskerprops={"color": "#555555", "linewidth": 0.9},
        capprops={"color": "#555555", "linewidth": 0.9},
    )
    for patch in box["boxes"]:
        patch.set_alpha(0.22)

    rng = np.random.default_rng(1234)
    for pos, values in zip(positions, data):
        jitter = rng.uniform(-0.08, 0.08, len(values))
        ax.scatter(
            np.full(len(values), pos) + jitter,
            values,
            s=8,
            color="#333333",
            alpha=0.35,
            linewidth=0,
            zorder=3,
        )

    for _, row in pivot.dropna(how="all").iterrows():
        available = row.dropna()
        if len(available) == 2:
            ax.plot(
                positions,
                available.to_numpy(),
                color="#777777",
                alpha=0.16,
                lw=0.45,
                zorder=1,
            )

    means = np.array([series.mean() if len(series) else np.nan for series in data])
    ax.plot(
        positions,
        means,
        color=color,
        marker="o",
        markersize=3.8,
        lw=1.4,
        zorder=4,
    )
    add_panel_title(
        ax,
        f"Pre-PC raw citations\nmean: $t_{{-2}}$ {means[0]:.1f}; $t_{{-1}}$ {means[1]:.1f}",
    )
    ax.set_xticks(positions)
    ax.set_xticklabels([r"$t_{-2}$", r"$t_{-1}$"])
    ax.set_xlabel("Pre-PC event time")
    ax.set_ylabel("Citations")
    ax.set_xlim(-0.45, 1.45)
    ax.set_ylim(bottom=-0.05)
    ax.grid(color="#DDDDDD", ls=":", lw=0.65)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def save_figure(fig, artifact_path, report_path=None):
    fig.savefig(artifact_path, bbox_inches="tight")
    print(f"wrote {artifact_path.relative_to(PROJECT)}")
    if report_path is not None:
        fig.savefig(report_path, bbox_inches="tight")
        print(f"wrote {report_path.relative_to(PROJECT)}")

## 7. Main text ICFP selection sensitivity figures

In [8]:
def main_text_sample_specs():
    return [
        {
            "sample": "all_isolated_balanced_event_units",
            "sample_label": "All event units",
            "title_line_1": "All event units",
            "title_line_2": "Full sample",
            "selector": lambda frame: frame,
            "seed": 1234,
        },
        {
            "sample": "all_isolated_balanced_career_age_20_plus",
            "sample_label": "All event units, career age 20+ years",
            "title_line_1": "All event units",
            "title_line_2": "Career age 20+ years",
            "selector": lambda frame: frame.loc[
                frame["career_age_at_first_pc"].ge(20)
            ],
            "seed": 1234,
        },
        {
            "sample": "no_earlier_broad_service_career_age_0_9",
            "sample_label": "No earlier service, career age 0-9 years",
            "title_line_1": "No earlier service",
            "title_line_2": "Career age 0-9 years",
            "selector": lambda frame: frame.loc[
                broad_first_service_mask(frame)
                & frame["career_age_at_first_pc"].lt(10)
            ],
            "seed": 1234,
        },
        {
            "sample": "no_earlier_broad_service_career_age_10_19",
            "sample_label": "No earlier service, career age 10-19 years",
            "title_line_1": "No earlier service",
            "title_line_2": "Career age 10-19 years",
            "selector": lambda frame: frame.loc[
                broad_first_service_mask(frame)
                & frame["career_age_at_first_pc"].ge(10)
                & frame["career_age_at_first_pc"].lt(20)
            ],
            "seed": 1234,
        },
    ]


def build_main_text_event_study_figure(rows, value_col, value_scale, ylabel, seed_offset=1234):
    icfp_rows = rows.loc[rows["conference"].eq("ICFP")].copy()
    summary_pieces = []
    plot_rows_by_sample = {}

    for spec in main_text_sample_specs():
        sample_rows = spec["selector"](icfp_rows).copy()
        plot_rows_by_sample[spec["sample"]] = sample_rows
        summary = bootstrap_summary(
            sample_rows,
            value_col,
            seed=1234,
        )
        if summary.empty:
            continue
        summary.insert(0, "conference", "ICFP")
        summary.insert(1, "sample", spec["sample"])
        summary.insert(2, "sample_label", spec["sample_label"])
        summary.insert(3, "value_scale", value_scale)
        summary_pieces.append(summary)

    summary_table = (
        pd.concat(summary_pieces, ignore_index=True)
        if summary_pieces
        else pd.DataFrame()
    )

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(9.0, 6.0),
        sharex=True,
        sharey=True,
    )
    axes = axes.ravel()
    color = CONFERENCE_COLORS["ICFP"]

    for idx, (ax, spec) in enumerate(zip(axes, main_text_sample_specs()), start=1):
        sample_summary = summary_table.loc[
            summary_table["sample"].eq(spec["sample"])
        ].copy()
        n_units = int(plot_rows_by_sample[spec["sample"]]["event_unit_id"].nunique())
        title = (
            f"{spec['title_line_1']}\n"
            f"{spec['title_line_2']} (N={n_units})"
        )
        draw_event_axis(
            ax,
            sample_summary,
            color,
            f"({chr(96 + idx)}) {title}",
            ylabel=ylabel,
            show_ylabel=idx in {1, 3},
            show_xlabel=idx in {3, 4},
        )
        ax.title.set_fontsize(7.8)
        ax.xaxis.label.set_size(8.0)
        ax.yaxis.label.set_size(8.0)
        ax.tick_params(axis="both", labelsize=7.5)

    handles = [
        plt.Line2D([0], [0], color="black", lw=0.85, ls="--"),
        plt.Line2D(
            [0],
            [0],
            color="black",
            lw=0.9,
            marker="_",
            markersize=8,
            linestyle="none",
        ),
    ]
    labels = ["first PC year", r"95\% bootstrap CI"]
    axes[0].legend(
        handles,
        labels,
        loc="lower left",
        frameon=False,
        fontsize=7.2,
        handlelength=1.6,
        labelspacing=0.25,
        borderaxespad=0.35,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.97], w_pad=1.8, h_pad=1.8)
    return summary_table, fig


log_summary, log_fig = build_main_text_event_study_figure(
    event_rows,
    value_col="delta_from_t_minus_2_log10_citations",
    value_scale="log10_citations_plus_one_t_minus_2_reference",
    ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$",
    seed_offset=1234,
)
raw_summary, raw_fig = build_main_text_event_study_figure(
    event_rows,
    value_col="delta_from_t_minus_2_raw_citations",
    value_scale="raw_citation_count_t_minus_2_reference",
    ylabel=r"$\Delta$ citations from $t_{-2}$",
    seed_offset=1234,
)

log_summary.to_csv(T_MINUS_2_LOG_SUMMARY_OUT, index=False)
raw_summary.to_csv(T_MINUS_2_RAW_SUMMARY_OUT, index=False)
save_figure(log_fig, T_MINUS_2_LOG_FIGURE_OUT, REPORT_MAIN_EVENT_LOG_FIGURE_OUT)
save_figure(raw_fig, T_MINUS_2_RAW_FIGURE_OUT, REPORT_MAIN_EVENT_RAW_FIGURE_OUT)
plt.close(log_fig)
plt.close(raw_fig)

display(log_summary)
display(raw_summary)

wrote step_4_artifacts/main_text_figures/t_minus_icfp_log.pdf
wrote report_latex/figures/event_study_icfp_log.pdf
wrote step_4_artifacts/main_text_figures/t_minus_icfp_raw.pdf
wrote report_latex/figures/event_study_icfp_raw.pdf


,conference,sample,sample_label,value_scale,event_time,mean,ci_lower,ci_upper,n_event_units
0,ICFP,all_isolated_balanced_event_units,All event units,log10_citations_plus_one_t_minus_2_reference,-2,0.000000,0.000000,0.000000,106
1,ICFP,all_isolated_balanced_event_units,All event units,log10_citations_plus_one_t_minus_2_reference,-1,0.033995,-0.036001,0.106772,106
2,ICFP,all_isolated_balanced_event_units,All event units,log10_citations_plus_one_t_minus_2_reference,0,0.099706,0.029950,0.171365,106
3,ICFP,all_isolated_balanced_event_units,All event units,log10_citations_plus_one_t_minus_2_reference,1,0.042467,-0.038236,0.119531,106
4,ICFP,all_isolated_balanced_event_units,All event units,log10_citations_plus_one_t_minus_2_reference,2,0.047811,-0.023998,0.118806,106
5,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",log10_citations_plus_one_t_minus_2_reference,-2,0.000000,0.000000,0.000000,39
6,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",log10_citations_plus_one_t_minus_2_reference,-1,0.065553,-0.053114,0.200206,39
7,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",log10_citations_plus_one_t_minus_2_reference,0,0.059363,-0.066877,0.186244,39
8,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",log10_citations_plus_one_t_minus_2_reference,1,-0.007384,-0.129647,0.111377,39
9,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",log10_citations_plus_one_t_minus_2_reference,2,-0.075975,-0.193938,0.050528,39


,conference,sample,sample_label,value_scale,event_time,mean,ci_lower,ci_upper,n_event_units
0,ICFP,all_isolated_balanced_event_units,All event units,raw_citation_count_t_minus_2_reference,-2,0.000000,0.000000,0.000000,106
1,ICFP,all_isolated_balanced_event_units,All event units,raw_citation_count_t_minus_2_reference,-1,0.226415,-0.641745,1.085142,106
2,ICFP,all_isolated_balanced_event_units,All event units,raw_citation_count_t_minus_2_reference,0,0.462264,-0.396226,1.292453,106
3,ICFP,all_isolated_balanced_event_units,All event units,raw_citation_count_t_minus_2_reference,1,-0.122642,-1.264387,0.924528,106
4,ICFP,all_isolated_balanced_event_units,All event units,raw_citation_count_t_minus_2_reference,2,0.320755,-0.528302,1.150943,106
5,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",raw_citation_count_t_minus_2_reference,-2,0.000000,0.000000,0.000000,39
6,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",raw_citation_count_t_minus_2_reference,-1,0.487179,-1.282051,2.257692,39
7,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",raw_citation_count_t_minus_2_reference,0,-0.102564,-2.051282,1.820513,39
8,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",raw_citation_count_t_minus_2_reference,1,-1.307692,-3.692308,0.538462,39
9,ICFP,all_isolated_balanced_career_age_20_plus,"All event units, career age 20+ years",raw_citation_count_t_minus_2_reference,2,-0.974359,-2.435897,0.461538,39


## 8. Conference specific appendix checks

In [9]:
def slugify(value):
    return (
        value.lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("-", "_")
    )


event_unit_summary = (
    event_rows.sort_values(["event_unit_id", "event_time"])
    .drop_duplicates("event_unit_id")
    [
        [
            "event_unit_id",
            "name",
            "conference",
            "plot_group",
            "first_pc_year_conference",
            "career_age_at_first_pc",
            "h_index",
            "h_index_fetch_date",
            "baseline_raw_citations",
        ]
    ]
    .copy()
)

h_index_dates = (
    event_unit_summary["h_index_fetch_date"]
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
)
h_index_date_label = h_index_dates[-1] if len(h_index_dates) else "unknown date"

max_career_age = event_unit_summary["career_age_at_first_pc"].max()
career_age_upper = int(np.ceil(max_career_age / 5) * 5)
career_age_upper = max(10, career_age_upper)
career_age_bins = list(range(0, career_age_upper + 5, 5))

max_h_index = event_unit_summary["h_index"].max()
h_index_upper = int(np.ceil(max_h_index / 10) * 10)
h_index_upper = max(10, h_index_upper)
h_index_bins = list(range(0, h_index_upper + 10, 10))


sample_definitions = [
    {
        "sample_key": "all_balanced_event_units",
        "page_title": "All researchers",
        "row_label": "All researchers",
        "selector": lambda frame: frame,
        "seed_base": 1234,
    },
    {
        "sample_key": "no_earlier_broad_service_evidence",
        "page_title": "No earlier service",
        "row_label": "No earlier service",
        "selector": lambda frame: frame.loc[broad_first_service_mask(frame)],
        "seed_base": 1234,
    },
]

aggregate_sample_definitions = [
    sample_definitions[0],
    {
        "sample_key": "excluded_from_no_earlier_service",
        "page_title": "Excluded from no earlier service",
        "row_label": "Excluded from no earlier service",
        "selector": lambda frame: frame.loc[
            ~broad_first_service_mask(frame)
        ],
        "seed_base": 1234,
    },
    sample_definitions[1],
]


def draw_sample_page(
    *,
    plot_group,
    plot_group_rows,
    sample_definition,
    group_idx,
    page_idx,
    color,
    split_kind,
):
    sample_rows = sample_definition["selector"](plot_group_rows).copy()
    sample_units = sample_rows.drop_duplicates("event_unit_id")
    if split_kind != "fixed_buckets":
        raise ValueError(f"Unknown split_kind: {split_kind}")
    median_career_age = sample_units["career_age_at_first_pc"].median()
    career_age = sample_rows["career_age_at_first_pc"]
    cohorts = [
        ("all_in_sample", sample_definition["row_label"], sample_rows),
        (
            "career_age_0_9",
            "Career age 0-9 years",
            sample_rows.loc[career_age.lt(10)],
        ),
        (
            "career_age_10_19",
            "Career age 10-19 years",
            sample_rows.loc[career_age.ge(10) & career_age.lt(20)],
        ),
        (
            "career_age_20_plus",
            "Career age 20+ years",
            sample_rows.loc[career_age.ge(20)],
        ),
    ]

    fig, axes = plt.subplots(
        nrows=len(cohorts),
        ncols=4,
        figsize=(15.2, 2.75 * len(cohorts) + 0.55),
        gridspec_kw={"width_ratios": [2.15, 1.0, 1.0, 1.05]},
    )
    axes = np.atleast_2d(axes)
    figure_records = []

    for row_idx, (cohort_key, cohort_title, cohort_rows) in enumerate(cohorts):
        cohort_units = cohort_rows.drop_duplicates("event_unit_id")
        n_units = int(cohort_units.shape[0])
        summary = bootstrap_summary(
            cohort_rows,
            "delta_from_t_minus_2_log10_citations",
            seed=1234,
        )

        draw_event_axis(
            axes[row_idx, 0],
            summary,
            color,
            f"{cohort_title}  |  N={n_units}",
            ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$",
            show_ylabel=True,
            show_xlabel=row_idx == len(cohorts) - 1,
        )
        draw_hist_axis(
            axes[row_idx, 1],
            cohort_units["career_age_at_first_pc"],
            bins=career_age_bins,
            color=color,
            title="Career age at first PC",
            xlabel="Years",
            xlim=(0, career_age_upper),
        )
        draw_hist_axis(
            axes[row_idx, 2],
            cohort_units["h_index"],
            bins=h_index_bins,
            color=color,
            title="h-index",
            xlabel="h-index",
            xlim=(0, h_index_upper),
            subtitle=f"OpenAlex {h_index_date_label}",
        )
        draw_pre_pc_raw_axis(
            axes[row_idx, 3],
            cohort_rows,
            color=color,
        )

        for axis in axes[row_idx, :]:
            axis.title.set_fontsize(8.2)
            axis.xaxis.label.set_size(8.0)
            axis.yaxis.label.set_size(8.0)
            axis.tick_params(axis="both", labelsize=7.6)

        figure_records.append(
            {
                "plot_group": plot_group,
                "sample": sample_definition["sample_key"],
                "split_kind": split_kind,
                "cohort": cohort_key,
                "median_career_age": median_career_age,
                "n_event_units": n_units,
            }
        )

    fig.suptitle(
        f"{plot_group}: {sample_definition['page_title']}",
        fontsize=13,
        y=0.99,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.1, w_pad=1.8)
    return fig, figure_records


def draw_aggregate_comparison_page(
    *,
    plot_group_rows,
    color,
    split_kind,
):
    if split_kind != "fixed_buckets":
        raise ValueError(f"Unknown split_kind: {split_kind}")

    fig, axes = plt.subplots(
        nrows=len(aggregate_sample_definitions),
        ncols=4,
        figsize=(15.2, 2.75 * len(aggregate_sample_definitions) + 0.55),
        gridspec_kw={"width_ratios": [2.15, 1.0, 1.0, 1.05]},
    )
    axes = np.atleast_2d(axes)
    figure_records = []

    for row_idx, sample_definition in enumerate(aggregate_sample_definitions):
        sample_rows = sample_definition["selector"](plot_group_rows).copy()
        sample_units = sample_rows.drop_duplicates("event_unit_id")
        n_units = int(sample_units.shape[0])
        summary = bootstrap_summary(
            sample_rows,
            "delta_from_t_minus_2_log10_citations",
            seed=1234,
        )

        draw_event_axis(
            axes[row_idx, 0],
            summary,
            color,
            f"{sample_definition['row_label']}  |  N={n_units}",
            ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$",
            show_ylabel=True,
            show_xlabel=row_idx == len(aggregate_sample_definitions) - 1,
        )
        draw_hist_axis(
            axes[row_idx, 1],
            sample_units["career_age_at_first_pc"],
            bins=career_age_bins,
            color=color,
            title="Career age at first PC",
            xlabel="Years",
            xlim=(0, career_age_upper),
        )
        draw_hist_axis(
            axes[row_idx, 2],
            sample_units["h_index"],
            bins=h_index_bins,
            color=color,
            title="h-index",
            xlabel="h-index",
            xlim=(0, h_index_upper),
            subtitle=f"OpenAlex {h_index_date_label}",
        )
        draw_pre_pc_raw_axis(
            axes[row_idx, 3],
            sample_rows,
            color=color,
        )

        for axis in axes[row_idx, :]:
            axis.title.set_fontsize(8.2)
            axis.xaxis.label.set_size(8.0)
            axis.yaxis.label.set_size(8.0)
            axis.tick_params(axis="both", labelsize=7.6)

        figure_records.append(
            {
                "plot_group": "Aggregate",
                "sample": sample_definition["sample_key"],
                "split_kind": split_kind,
                "cohort": "all_in_sample",
                "median_career_age": sample_units["career_age_at_first_pc"].median(),
                "n_event_units": n_units,
            }
        )

    fig.suptitle(
        "Aggregate: all researchers, excluded complement, and no earlier service",
        fontsize=13,
        y=0.99,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.1, w_pad=1.8)
    return fig, figure_records


observed_plot_groups = set(event_rows["plot_group"])
available_plot_groups = ["Aggregate"] + [
    group
    for group in ["POPL", "ICFP", "OOPSLA", "PLDI before 2021", "PLDI 2021 onward"]
    if group in observed_plot_groups
]
plot_group_slugs = {group: slugify(group) for group in available_plot_groups}

figure_families = [
    {"split_kind": "fixed_buckets", "file_suffix": "fixed_buckets"},
]

all_records = []
for family in figure_families:
    for group_idx, plot_group in enumerate(available_plot_groups):
        if plot_group == "Aggregate":
            plot_group_rows = event_rows.loc[
                event_rows["plot_group"].isin(aggregate_groups)
            ].copy()
        else:
            plot_group_rows = event_rows.loc[
                event_rows["plot_group"].eq(plot_group)
            ].copy()
        color = CONFERENCE_COLORS.get(plot_group, MAIN_COLOR)
        figure_out = (
            T_MINUS_EVENT_FIGURES
            / f"{plot_group_slugs[plot_group]}_{family['file_suffix']}.pdf"
        )

        with PdfPages(figure_out) as pdf:
            if plot_group == "Aggregate":
                group_sample_definitions = aggregate_sample_definitions
            else:
                group_sample_definitions = sample_definitions

            for page_idx, sample_definition in enumerate(group_sample_definitions):
                fig, figure_records = draw_sample_page(
                    plot_group=plot_group,
                    plot_group_rows=plot_group_rows,
                    sample_definition=sample_definition,
                    group_idx=group_idx,
                    page_idx=page_idx,
                    color=color,
                    split_kind=family["split_kind"],
                )
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
                all_records.extend(figure_records)
        print(f"wrote {figure_out.relative_to(PROJECT)}")

aggregate_rows = event_rows.loc[
    event_rows["plot_group"].isin(aggregate_groups)
].copy()
aggregate_color = CONFERENCE_COLORS.get("Aggregate", MAIN_COLOR)
aggregate_comparison_fig, _ = draw_aggregate_comparison_page(
    plot_group_rows=aggregate_rows,
    color=aggregate_color,
    split_kind="fixed_buckets",
)
save_figure(
    aggregate_comparison_fig,
    AGGREGATE_COMPARISON_FIGURE_OUT,
    REPORT_AGGREGATE_COMPARISON_FIGURE_OUT,
)
plt.close(aggregate_comparison_fig)

figure_records = pd.DataFrame(all_records)
figure_records.to_csv(T_MINUS_2_FIGURE_RECORDS_OUT, index=False)
display(figure_records)

wrote step_4_artifacts/figures_t_minus_event_study/aggregate_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/popl_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/icfp_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/oopsla_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/pldi_before_2021_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/pldi_2021_onward_fixed_buckets.pdf
wrote step_4_artifacts/figures_t_minus_event_study/aggregate_sample_comparison.pdf
wrote report_latex/figures/aggregate_sample_comparison.pdf


,plot_group,sample,split_kind,cohort,median_career_age,n_event_units
0,Aggregate,all_balanced_event_units,fixed_buckets,all_in_sample,16.0,349
1,Aggregate,all_balanced_event_units,fixed_buckets,career_age_0_9,16.0,60
2,Aggregate,all_balanced_event_units,fixed_buckets,career_age_10_19,16.0,159
3,Aggregate,all_balanced_event_units,fixed_buckets,career_age_20_plus,16.0,130
4,Aggregate,excluded_from_no_earlier_service,fixed_buckets,all_in_sample,18.0,248
5,Aggregate,excluded_from_no_earlier_service,fixed_buckets,career_age_0_9,18.0,34
6,Aggregate,excluded_from_no_earlier_service,fixed_buckets,career_age_10_19,18.0,102
7,Aggregate,excluded_from_no_earlier_service,fixed_buckets,career_age_20_plus,18.0,112
8,Aggregate,no_earlier_broad_service_evidence,fixed_buckets,all_in_sample,14.0,101
9,Aggregate,no_earlier_broad_service_evidence,fixed_buckets,career_age_0_9,14.0,26


## 10. Save the re referenced event rows

In [10]:
event_rows.to_parquet(T_MINUS_2_ROWS_OUT, index=False)

print("Outputs:")
for path in [
    T_MINUS_2_ROWS_OUT,
    T_MINUS_2_LOG_SUMMARY_OUT,
    T_MINUS_2_RAW_SUMMARY_OUT,
    T_MINUS_2_FIGURE_RECORDS_OUT,
    T_MINUS_2_LOG_FIGURE_OUT,
    T_MINUS_2_RAW_FIGURE_OUT,
    AGGREGATE_COMPARISON_FIGURE_OUT,
]:
    print(path.relative_to(PROJECT))

Outputs:
step_4_data/prepared/t_minus_2_reference_event_window_rows.parquet
step_4_artifacts/summary_tables/tminus2_icfp_log.csv
step_4_artifacts/summary_tables/tminus2_icfp_raw.csv
step_4_artifacts/summary_tables/tminus2_figure_records.csv
step_4_artifacts/main_text_figures/t_minus_icfp_log.pdf
step_4_artifacts/main_text_figures/t_minus_icfp_raw.pdf
step_4_artifacts/figures_t_minus_event_study/aggregate_sample_comparison.pdf


## 11. Figures produced

The table below lists every PDF figure written by this notebook. The
previews show the same files from the artifact folder.

In [11]:
figure_outputs = [
    ("ICFP main-text log-scale figure", MAIN_TEXT_FIGURES / "t_minus_icfp_log.pdf"),
    ("ICFP main-text raw-count figure", MAIN_TEXT_FIGURES / "t_minus_icfp_raw.pdf"),
    ("Aggregate fixed career-age buckets", T_MINUS_EVENT_FIGURES / "aggregate_fixed_buckets.pdf"),
    ("Aggregate compact sample comparison", T_MINUS_EVENT_FIGURES / "aggregate_sample_comparison.pdf"),
    ("POPL fixed career-age buckets", T_MINUS_EVENT_FIGURES / "popl_fixed_buckets.pdf"),
    ("ICFP fixed career-age buckets", T_MINUS_EVENT_FIGURES / "icfp_fixed_buckets.pdf"),
    ("OOPSLA fixed career-age buckets", T_MINUS_EVENT_FIGURES / "oopsla_fixed_buckets.pdf"),
    ("PLDI before 2021 fixed career-age buckets", T_MINUS_EVENT_FIGURES / "pldi_before_2021_fixed_buckets.pdf"),
    ("PLDI 2021 onward fixed career-age buckets", T_MINUS_EVENT_FIGURES / "pldi_2021_onward_fixed_buckets.pdf"),
]

figure_table = pd.DataFrame(
    {
        "figure": label,
        "path": str(path.relative_to(PROJECT)),
        "exists": path.exists(),
    }
    for label, path in figure_outputs
)
display(figure_table)

notebook_root = Path("..") / ".."

def show_pdf(label, path, height=620):
    rel_project_path = path.relative_to(PROJECT)
    display(Markdown(f"**{label}**  \n`{rel_project_path}`"))
    if path.exists():
        display(IFrame(src=str(notebook_root / rel_project_path), width="100%", height=height))
    else:
        display(Markdown("Missing figure file."))

for label, path in figure_outputs:
    show_pdf(label, path)

,figure,path,exists
0,ICFP main-text log-scale figure,step_4_artifacts/main_text_figures/t_minus_icf...,True
1,ICFP main-text raw-count figure,step_4_artifacts/main_text_figures/t_minus_icf...,True
2,Aggregate fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/a...,True
3,Aggregate compact sample comparison,step_4_artifacts/figures_t_minus_event_study/a...,True
4,POPL fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/p...,True
5,ICFP fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/i...,True
6,OOPSLA fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/o...,True
7,PLDI before 2021 fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/p...,True
8,PLDI 2021 onward fixed career-age buckets,step_4_artifacts/figures_t_minus_event_study/p...,True


**ICFP main-text log-scale figure**  
`step_4_artifacts/main_text_figures/t_minus_icfp_log.pdf`

**ICFP main-text raw-count figure**  
`step_4_artifacts/main_text_figures/t_minus_icfp_raw.pdf`

**Aggregate fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/aggregate_fixed_buckets.pdf`

**Aggregate compact sample comparison**  
`step_4_artifacts/figures_t_minus_event_study/aggregate_sample_comparison.pdf`

**POPL fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/popl_fixed_buckets.pdf`

**ICFP fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/icfp_fixed_buckets.pdf`

**OOPSLA fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/oopsla_fixed_buckets.pdf`

**PLDI before 2021 fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/pldi_before_2021_fixed_buckets.pdf`

**PLDI 2021 onward fixed career-age buckets**  
`step_4_artifacts/figures_t_minus_event_study/pldi_2021_onward_fixed_buckets.pdf`